In [1]:
import numpy as np
from typing import List, Tuple, Optional, Callable

class Variable:
    """A node in the computation graph."""
    
    def __init__(self, data: np.ndarray, parents: Tuple['Variable', ...] = (), 
                 operation: str = '', grad_fn: Optional[Callable] = None):
        """
        Args:
            data: The actual value stored in this variable
            parents: The variables that were used to create this one
            operation: String describing the operation (for visualization)
            grad_fn: Function to compute gradients during backprop
        """
        self.data = np.array(data, dtype=np.float32)
        self.parents = parents
        self.operation = operation
        self.grad_fn = grad_fn
        self.grad = np.zeros_like(self.data)
        
    def __repr__(self):
        return f"Variable(data={self.data}, op='{self.operation}')"
    
    # Operator overloading - this is where the magic happens!
    def __add__(self, other):
        other = other if isinstance(other, Variable) else Variable(other)
        result = Variable(
            data=self.data + other.data,
            parents=(self, other),
            operation='add',
            grad_fn=lambda grad: (grad, grad)  # Gradient flows equally to both parents
        )
        return result
    
    def __mul__(self, other):
        other = other if isinstance(other, Variable) else Variable(other)
        result = Variable(
            data=self.data * other.data,
            parents=(self, other),
            operation='mul',
            grad_fn=lambda grad: (grad * other.data, grad * self.data)
        )
        return result
    
    def __pow__(self, power):
        result = Variable(
            data=self.data ** power,
            parents=(self,),
            operation=f'pow({power})',
            grad_fn=lambda grad: (grad * power * self.data ** (power - 1),)
        )
        return result
    
    def __sub__(self, other):
        other = other if isinstance(other, Variable) else Variable(other)
        return self + (other * -1)
    
    def __truediv__(self, other):
        other = other if isinstance(other, Variable) else Variable(other)
        return self * (other ** -1)
    
    # Right-hand operators (for scalar * Variable, etc.)
    def __radd__(self, other):
        return self.__add__(other)
    
    def __rmul__(self, other):
        return self.__mul__(other)
    
    def backward(self):
        """Compute gradients via backpropagation through the computation graph."""
        # Topological sort of the computation graph
        topo_order = []
        visited = set()
        
        def build_topo(node):
            if node not in visited:
                visited.add(node)
                for parent in node.parents:
                    build_topo(parent)
                topo_order.append(node)
        
        build_topo(self)
        
        # Backpropagate gradients
        self.grad = np.ones_like(self.data)
        for node in reversed(topo_order):
            if node.grad_fn and node.parents:
                grads = node.grad_fn(node.grad)
                for parent, grad in zip(node.parents, grads):
                    parent.grad += grad
    
    def visualize_graph(self, indent=0):
        """Print the computation graph structure."""
        print("  " * indent + f"{self.operation or 'input'}: {self.data}")
        for parent in self.parents:
            parent.visualize_graph(indent + 1)


def tanh(x: Variable) -> Variable:
    """Hyperbolic tangent activation function."""
    t = np.tanh(x.data)
    result = Variable(
        data=t,
        parents=(x,),
        operation='tanh',
        grad_fn=lambda grad: (grad * (1 - t**2),)
    )
    return result


def relu(x: Variable) -> Variable:
    """ReLU activation function."""
    result = Variable(
        data=np.maximum(0, x.data),
        parents=(x,),
        operation='relu',
        grad_fn=lambda grad: (grad * (x.data > 0),)
    )
    return result


In [2]:
print("=" * 60)
print("Example 1: Simple computation graph")
print("=" * 60)

# Define inputs (leaf nodes)
x = Variable(2.0)
y = Variable(3.0)

# Build computation graph through natural equations
z = x * y + x ** 2
w = z * 2 + y

print(f"x = {x.data}")
print(f"y = {y.data}")
print(f"z = x * y + x^2 = {z.data}")
print(f"w = z * 2 + y = {w.data}")

print("\nComputation graph for w:")
w.visualize_graph()

# Compute gradients
print("\nComputing gradients...")
w.backward()
print(f"dw/dx = {x.grad}")
print(f"dw/dy = {y.grad}")

print("\n" + "=" * 60)
print("Example 2: Neural network-style computation")
print("=" * 60)

# Simple neuron: y = tanh(w1*x1 + w2*x2 + b)
x1 = Variable(1.0)
x2 = Variable(2.0)
w1 = Variable(0.5)
w2 = Variable(-0.3)
b = Variable(0.1)

# Forward pass
linear = w1 * x1 + w2 * x2 + b
output = tanh(linear)

print(f"Inputs: x1={x1.data}, x2={x2.data}")
print(f"Weights: w1={w1.data}, w2={w2.data}, b={b.data}")
print(f"Linear: {linear.data}")
print(f"Output: {output.data}")

print("\nComputation graph:")
output.visualize_graph()

# Backward pass
output.backward()
print("\nGradients:")
print(f"dOutput/dw1 = {w1.grad}")
print(f"dOutput/dw2 = {w2.grad}")
print(f"dOutput/db = {b.grad}")

print("\n" + "=" * 60)
print("Example 3: More complex expression")
print("=" * 60)

a = Variable(2.0)
b = Variable(3.0)
c = Variable(4.0)

# Complex expression built naturally
result = (a + b) * c / (a - b) + a ** 3

print(f"Result = (a + b) * c / (a - b) + a^3")
print(f"       = ({a.data} + {b.data}) * {c.data} / ({a.data} - {b.data}) + {a.data}^3")
print(f"       = {result.data}")

result.backward()
print(f"\nGradients: da={a.grad}, db={b.grad}, dc={c.grad}")

Example 1: Simple computation graph
x = 2.0
y = 3.0
z = x * y + x^2 = 10.0
w = z * 2 + y = 23.0

Computation graph for w:
add: 23.0
  mul: 20.0
    add: 10.0
      mul: 6.0
        input: 2.0
        input: 3.0
      pow(2): 4.0
        input: 2.0
    input: 2.0
  input: 3.0

Computing gradients...
dw/dx = 14.0
dw/dy = 5.0

Example 2: Neural network-style computation
Inputs: x1=1.0, x2=2.0
Weights: w1=0.5, w2=-0.30000001192092896, b=0.10000000149011612
Linear: -2.2351741790771484e-08
Output: -2.2351741790771484e-08

Computation graph:
tanh: -2.2351741790771484e-08
  add: -2.2351741790771484e-08
    add: -0.10000002384185791
      mul: 0.5
        input: 0.5
        input: 1.0
      mul: -0.6000000238418579
        input: -0.30000001192092896
        input: 2.0
    input: 0.10000000149011612

Gradients:
dOutput/dw1 = 1.0
dOutput/dw2 = 2.0
dOutput/db = 1.0

Example 3: More complex expression
Result = (a + b) * c / (a - b) + a^3
       = (2.0 + 3.0) * 4.0 / (2.0 - 3.0) + 2.0^3
       = -1